# 01 — Source Validation

Validate connectivity to all four openFDA data sources and confirm key fields are present.

In [ ]:
import sys

sys.path.insert(0, "..")

from src.api import FDAClient
from src.config import (
    DATE_END,
    DATE_START,
    ENDPOINT_510K,
    ENDPOINT_ADVERSE_EVENTS,
    ENDPOINT_CLASSIFICATION,
    ENDPOINT_RECALL,
)

client = FDAClient()
print(f"API key configured: {bool(client.api_key)}")
print(f"Rate limit: {client.rate_limit} req/min")

## 1. Adverse Events (MAUDE)

In [ ]:
ae_data = client.fetch_page(ENDPOINT_ADVERSE_EVENTS, limit=5)
ae_total = ae_data["meta"]["results"]["total"]
print(f"Total adverse event records: {ae_total:,}")

# Verify key fields
sample = ae_data["results"][0]
ae_fields = ["mdr_report_key", "date_received", "device", "patient"]
for field in ae_fields:
    present = field in sample
    print(f"  {field}: {'✓' if present else '✗'}")

print(f"\nSample record keys: {list(sample.keys())}")

## 2. Device Classification

In [ ]:
cls_data = client.fetch_page(ENDPOINT_CLASSIFICATION, limit=5)
cls_total = cls_data["meta"]["results"]["total"]
print(f"Total classification records: {cls_total:,}")

sample = cls_data["results"][0]
cls_fields = ["product_code", "device_name", "device_class", "medical_specialty_description"]
for field in cls_fields:
    present = field in sample
    print(f"  {field}: {'✓' if present else '✗'}")

print(f"\nSample record keys: {list(sample.keys())}")

## 3. Recalls / Enforcement

In [ ]:
recall_search = 'product_type:"Devices"'
recall_data = client.fetch_page(ENDPOINT_RECALL, search=recall_search, limit=5)
recall_total = recall_data["meta"]["results"]["total"]
print(f"Total device recall records: {recall_total:,}")

sample = recall_data["results"][0]
recall_fields = ["recall_number", "recall_initiation_date", "classification", "recalling_firm"]
for field in recall_fields:
    present = field in sample
    print(f"  {field}: {'✓' if present else '✗'}")

print(f"\nSample record keys: {list(sample.keys())}")

## 4. 510(k) Clearances

In [ ]:
k510_data = client.fetch_page(ENDPOINT_510K, limit=5)
k510_total = k510_data["meta"]["results"]["total"]
print(f"Total 510(k) records: {k510_total:,}")

sample = k510_data["results"][0]
k510_fields = ["k_number", "decision_date", "product_code", "applicant"]
for field in k510_fields:
    present = field in sample
    print(f"  {field}: {'✓' if present else '✗'}")

print(f"\nSample record keys: {list(sample.keys())}")

## 5. Bulk Download Check

In [ ]:
import requests

download_resp = requests.get("https://api.fda.gov/download.json", timeout=30)
download_index = download_resp.json()

partitions = download_index["results"]["device"]["event"]["partitions"]
print(f"Total device event partitions: {len(partitions)}")

# Filter to 2019-2025 window
from src.extraction.adverse_events import AdverseEventExtractor

filtered = AdverseEventExtractor._filter_partitions(partitions, 2019, 2025)
print(f"Partitions in 2019-2025 window: {len(filtered)}")

total_size_mb = sum(float(p.get("size_mb", 0)) for p in filtered)
print(f"Estimated total download size: {total_size_mb:,.0f} MB ({total_size_mb / 1024:.1f} GB)")

## 6. Summary

In [ ]:
import pandas as pd

summary = pd.DataFrame(
    [
        {
            "Source": "Adverse Events (MAUDE)",
            "Total Records": ae_total,
            "Key Fields": ", ".join(ae_fields),
            "Date Coverage": f"{DATE_START} to {DATE_END}",
        },
        {
            "Source": "Device Classification",
            "Total Records": cls_total,
            "Key Fields": ", ".join(cls_fields),
            "Date Coverage": "All (no date filter)",
        },
        {
            "Source": "Recalls / Enforcement",
            "Total Records": recall_total,
            "Key Fields": ", ".join(recall_fields),
            "Date Coverage": f"{DATE_START} to {DATE_END}",
        },
        {
            "Source": "510(k) Clearances",
            "Total Records": k510_total,
            "Key Fields": ", ".join(k510_fields),
            "Date Coverage": f"{DATE_START} to {DATE_END}",
        },
    ]
)

print(summary.to_string(index=False))